In [1]:
import matplotlib.pyplot as plt
import numpy as np
import mitsuba as mi
import pyvista as pv
import sionna
import tensorflow as tf
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, PathSolver, Camera
from scipy.optimize import minimize
import h5py
from pathlib import Path
import sys

In [2]:
# ============================================================================
# SYSTEM PARAMETERS AND CLASS DEFINITIONS
# ============================================================================

# --- SYSTEM PARAMETERS ---
CARRIER_FREQUENCY = 2.4e9  
BANDWIDTH = 20e6           
NUM_SUBCARRIERS = 64       
SUBCARRIER_SPACING = BANDWIDTH / NUM_SUBCARRIERS
WAVELENGTH = 3e8 / CARRIER_FREQUENCY
NUM_ANTENNAS_RX = 4

# --- Define subcarrier frequencies ---
subcarrier_indices = np.arange(NUM_SUBCARRIERS)
subcarriers = CARRIER_FREQUENCY + (subcarrier_indices - NUM_SUBCARRIERS/2) * SUBCARRIER_SPACING

# --- Import for peak detection ---
from scipy.signal import find_peaks


class CSIAngleEstimator:
    """Angle of Arrival estimator using MUSIC algorithm with planar array support"""
    def __init__(self, num_antennas, wavelength, array_type='planar'):
        self.num_antennas = num_antennas
        self.wavelength = wavelength
        self.antenna_spacing = wavelength / 2
        self.k = 2 * np.pi / wavelength
        self.array_type = array_type
        
        # For 2x2 planar array: define element positions
        if array_type == 'planar' and num_antennas == 4:
            # 2x2 grid positions: (0,0), (1,0), (0,1), (1,1)
            self.pos_x = np.array([0, 1, 0, 1]) * self.antenna_spacing
            self.pos_y = np.array([0, 0, 1, 1]) * self.antenna_spacing
        else:
            # Linear array fallback
            self.pos_x = np.arange(num_antennas) * self.antenna_spacing
            self.pos_y = np.zeros(num_antennas)
    
    def steering_vector(self, azimuth, elevation=0):
        """2D planar array steering vector"""
        # Phase shift based on planar geometry
        phase = self.k * (self.pos_x * np.cos(azimuth) * np.cos(elevation) + 
                          self.pos_y * np.sin(azimuth) * np.cos(elevation))
        return np.exp(1j * phase)
    
    def estimate_aoa_music(self, csi_matrix, num_sources=1, return_spectrum=False):
        # csi_matrix must be [Antennas, Subcarriers]
        # Forward covariance
        R_forward = csi_matrix @ csi_matrix.conj().T / csi_matrix.shape[1]
        
        # Backward covariance (for forward-backward averaging)
        J = np.fliplr(np.eye(self.num_antennas))  # Exchange matrix
        R_backward = J @ R_forward.conj() @ J
        
        # Forward-backward averaged covariance (more robust, better rank)
        R = (R_forward + R_backward) / 2
        
        eigenvalues, eigenvectors = np.linalg.eigh(R)
        idx = eigenvalues.argsort()[::-1]
        noise_subspace = eigenvectors[:, idx[num_sources:]]
        
        # Extended search range: full 360° (-π to π)
        angles = np.linspace(-np.pi, np.pi, 360)
        spectrum = np.zeros(len(angles))
        
        for i, angle in enumerate(angles):
            a = self.steering_vector(angle)
            spectrum[i] = 1.0 / (np.abs(
                a.conj() @ noise_subspace @ noise_subspace.conj().T @ a
            ) + 1e-10)
        
        peak_idx = np.argmax(spectrum)
        aoa_estimate = angles[peak_idx]
        
        if return_spectrum:
            return aoa_estimate, spectrum
        else:
            return aoa_estimate


class FTMRangeEstimator:
    """Range estimator from CSI phase slope (FTM mechanism) with first-arrival detection"""
    def __init__(self, speed_of_light=3e8):
        self.c = speed_of_light
    
    def estimate_range_from_phase(self, csi_subcarriers, subcarrier_spacing):
        phase = np.unwrap(np.angle(csi_subcarriers))
        subcarrier_idx = np.arange(len(phase))
        coeffs = np.polyfit(subcarrier_idx * subcarrier_spacing, phase, 1)
        phase_slope = coeffs[0]
        toa = -phase_slope / (2 * np.pi)
        return abs(self.c * toa)  # One-way delay from CSI, no /2
    
    def estimate_range_from_cir(self, delays, amplitudes, power_threshold_db=-20):
        """
        First-arrival ToA ranging: uses first path with significant power (LoS)
        instead of strongest path which could be NLOS reflection.
        """
        powers = np.abs(amplitudes) ** 2
        max_power = np.max(powers)
        threshold = max_power * (10 ** (power_threshold_db / 10))
        
        # Find first path above threshold (likely LoS)
        valid_mask = powers > threshold
        if np.any(valid_mask):
            valid_delays = delays[valid_mask]
            first_arrival_delay = np.min(valid_delays)
        else:
            # Fallback to minimum delay
            first_arrival_delay = np.min(delays)
        
        return self.c * first_arrival_delay


class BilaterationSolver:
    """Hybrid localization using range and angle (Weighted Least Squares)"""
    @staticmethod
    def hybrid_localization(anchor_positions, ranges, angles, distance_std=0.5, angle_std=0.1):
        # Ensure anchor_positions is a 2D array
        anchor_positions = np.asarray(anchor_positions)
        if anchor_positions.ndim == 1:
            anchor_positions = anchor_positions.reshape(1, -1)
        
        # Ensure ranges and angles are 1D arrays
        ranges = np.asarray(ranges).flatten()
        angles = np.asarray(angles).flatten()
        
        num_anchors = len(anchor_positions)
        distance_weight = 1 / (distance_std**2)
        angle_weight = 1 / (angle_std**2)
        
        def objective(pos):
            # Ensure pos is 1D
            pos = np.asarray(pos).flatten()
            
            # 1. Distance Errors (Ranging)
            error_d = 0.0
            for i in range(num_anchors):
                dist = np.linalg.norm(pos - anchor_positions[i])
                error_d += distance_weight * (ranges[i] - dist)**2
            
            # 2. Angle Errors (AoA)
            error_a = 0.0 
            
            for i in range(num_anchors):
                if not np.isnan(angles[i]):
                    # Extract x,y coordinates as scalars
                    pos_x = float(pos[0])
                    pos_y = float(pos[1])
                    anchor_x = float(anchor_positions[i, 0])
                    anchor_y = float(anchor_positions[i, 1])
                    
                    # Calculate direction FROM ANCHOR TO TARGET
                    dir_x = pos_x - anchor_x
                    dir_y = pos_y - anchor_y
                    
                    estimated_angle = np.arctan2(dir_y, dir_x)
                    angle_diff = float(angles[i]) - estimated_angle
                    
                    # Normalize angle difference to [-pi, pi]
                    angle_diff = np.arctan2(np.sin(angle_diff), np.cos(angle_diff))
                    error_a += angle_weight * (angle_diff**2)
            
            # Total Error
            total_error = error_d + error_a
            return float(total_error)
        
        # Initial guess: mean of anchor positions - ENSURE 1D
        x0 = np.mean(anchor_positions, axis=0).flatten()
        
        # Debug: verify x0 is 1D
        if x0.ndim != 1:
            raise ValueError(f"x0 has {x0.ndim} dimensions, shape {x0.shape}. Expected 1D array.")
        
        result = minimize(objective, x0, method='BFGS') 
        return result.x


# ============================================================================
# ENHANCED NLOS MITIGATION CLASSES
# ============================================================================

class EnhancedFTMRangeEstimator:
    """Range estimator with advanced NLOS detection and mitigation"""
    def __init__(self, speed_of_light=3e8):
        self.c = speed_of_light
    
    def estimate_range_from_cir(self, delays, amplitudes, power_threshold_db=-20):
        """Basic first-arrival ToA ranging (for compatibility)"""
        powers = np.abs(amplitudes) ** 2
        max_power = np.max(powers)
        threshold = max_power * (10 ** (power_threshold_db / 10))
        
        valid_mask = powers > threshold
        if np.any(valid_mask):
            valid_delays = delays[valid_mask]
            first_arrival_delay = np.min(valid_delays)
        else:
            first_arrival_delay = np.min(delays)
        
        return self.c * first_arrival_delay
    
    def detect_nlos(self, delays, amplitudes, kurtosis_threshold=3.0, rician_k_threshold=2.0):
        """
        NLOS detection using statistical channel characteristics.
        Returns: (is_nlos, nlos_probability, metrics)
        """
        powers = np.abs(amplitudes) ** 2
        powers_normalized = powers / (np.sum(powers) + 1e-12)
        
        # 1. Kurtosis-based detection (NLOS has higher kurtosis)
        mean_power = np.mean(powers)
        std_power = np.std(powers) + 1e-12
        kurtosis = np.mean((powers - mean_power)**4) / (std_power**4)
        
        # 2. Rician K-factor estimation (LOS has higher K)
        sorted_powers = np.sort(powers)[::-1]
        if len(sorted_powers) > 1:
            k_factor = sorted_powers[0] / (np.sum(sorted_powers[1:]) + 1e-12)
        else:
            k_factor = np.inf
        
        # 3. RMS delay spread (NLOS has larger spread)
        mean_delay = np.sum(delays * powers_normalized)
        rms_delay_spread = np.sqrt(np.sum((delays - mean_delay)**2 * powers_normalized))
        
        # 4. First-to-strongest path delay difference
        power_threshold = powers.max() * 0.01
        valid_indices = np.where(powers > power_threshold)[0]
        if len(valid_indices) > 0:
            first_path_idx = valid_indices[np.argmin(delays[valid_indices])]
        else:
            first_path_idx = np.argmin(delays)
        strongest_path_idx = np.argmax(powers)
        delay_diff = abs(delays[strongest_path_idx] - delays[first_path_idx])
        
        # NLOS probability based on metrics
        nlos_score = 0.0
        if kurtosis > kurtosis_threshold:
            nlos_score += 0.3
        if k_factor < rician_k_threshold:
            nlos_score += 0.3
        if delay_diff > 10e-9:  # >10ns difference suggests NLOS
            nlos_score += 0.4
        
        is_nlos = nlos_score > 0.5
        
        return is_nlos, nlos_score, {
            'kurtosis': kurtosis,
            'k_factor': k_factor,
            'rms_delay_spread': rms_delay_spread,
            'delay_diff': delay_diff
        }
    
    def estimate_range_nlos_mitigated(self, delays, amplitudes, power_threshold_db=-15):
        """
        NLOS-mitigated range estimation using multiple techniques.
        """
        powers = np.abs(amplitudes) ** 2
        
        # 1. Detect NLOS condition
        is_nlos, nlos_prob, metrics = self.detect_nlos(delays, amplitudes)
        
        # 2. First-arrival detection (baseline)
        threshold = np.max(powers) * (10 ** (power_threshold_db / 10))
        valid_mask = powers > threshold
        
        if np.any(valid_mask):
            valid_delays = delays[valid_mask]
            first_arrival_delay = np.min(valid_delays)
        else:
            first_arrival_delay = np.min(delays)
        
        # 3. If NLOS detected, apply additional mitigation
        if is_nlos:
            # Use lower threshold to catch weaker LOS component
            stricter_threshold = np.max(powers) * (10 ** (-25 / 10))
            strict_mask = powers > stricter_threshold
            if np.any(strict_mask):
                first_arrival_delay = np.min(delays[strict_mask])
            
            # Apply NLOS bias correction (empirical: NLOS adds ~0.5-2m positive bias)
            nlos_bias_correction = 0.5 * nlos_prob  # Scale by NLOS probability
            range_estimate = self.c * first_arrival_delay - nlos_bias_correction
        else:
            range_estimate = self.c * first_arrival_delay
        
        return max(0, range_estimate), is_nlos, nlos_prob


class EnhancedAoAEstimator:
    """AoA estimator with NLOS mitigation using spatial smoothing and robust peak detection"""
    def __init__(self, num_antennas, wavelength, array_type='planar'):
        self.num_antennas = num_antennas
        self.wavelength = wavelength
        self.antenna_spacing = wavelength / 2
        self.k = 2 * np.pi / wavelength
        self.array_type = array_type
        
        if array_type == 'planar' and num_antennas == 4:
            self.pos_x = np.array([0, 1, 0, 1]) * self.antenna_spacing
            self.pos_y = np.array([0, 0, 1, 1]) * self.antenna_spacing
        else:
            self.pos_x = np.arange(num_antennas) * self.antenna_spacing
            self.pos_y = np.zeros(num_antennas)
    
    def steering_vector(self, azimuth, elevation=0):
        """2D planar array steering vector"""
        phase = self.k * (self.pos_x * np.cos(azimuth) * np.cos(elevation) + 
                          self.pos_y * np.sin(azimuth) * np.cos(elevation))
        return np.exp(1j * phase)
    
    def estimate_aoa_music_nlos_robust(self, csi_matrix, delays=None, amplitudes=None, 
                                        num_sources=1, return_spectrum=False):
        """
        NLOS-robust MUSIC with spatial smoothing and multipath-aware processing.
        """
        M = csi_matrix.shape[0]  # Number of antennas
        N = csi_matrix.shape[1]  # Number of subcarriers
        
        # 1. Spatial smoothing for decorrelation (helps with coherent multipath)
        if M >= 3:
            L = max(2, M - 1)  # Subarray size
            R_smooth = np.zeros((L, L), dtype=complex)
            num_subarrays = M - L + 1
            
            for i in range(num_subarrays):
                sub_csi = csi_matrix[i:i+L, :]
                R_sub = sub_csi @ sub_csi.conj().T / N
                R_smooth += R_sub
            
            # Backward smoothing for better decorrelation
            J = np.fliplr(np.eye(L))
            R_smooth = (R_smooth + J @ R_smooth.conj() @ J) / (2 * num_subarrays)
            
            # Eigendecomposition on smoothed covariance
            eigenvalues, eigenvectors = np.linalg.eigh(R_smooth)
            effective_antennas = L
            
            # Update steering vector dimensions for subarray
            pos_x_sub = self.pos_x[:L]
            pos_y_sub = self.pos_y[:L]
        else:
            # Fall back to standard forward-backward averaging
            R_forward = csi_matrix @ csi_matrix.conj().T / N
            J = np.fliplr(np.eye(M))
            R_backward = J @ R_forward.conj() @ J
            R_smooth = (R_forward + R_backward) / 2
            eigenvalues, eigenvectors = np.linalg.eigh(R_smooth)
            effective_antennas = M
            pos_x_sub = self.pos_x
            pos_y_sub = self.pos_y
        
        idx = eigenvalues.argsort()[::-1]
        sorted_eigs = eigenvalues[idx]
        
        # 2. MDL-based source number estimation
        estimated_sources = 1
        for k in range(1, len(sorted_eigs)):
            noise_eigs = sorted_eigs[k:]
            if len(noise_eigs) > 0:
                noise_power = np.mean(noise_eigs) + 1e-12
                signal_power = sorted_eigs[k-1]
                if signal_power / noise_power < 2:
                    estimated_sources = max(1, k)
                    break
        
        num_sources = min(estimated_sources, num_sources, effective_antennas - 1)
        noise_subspace = eigenvectors[:, idx[num_sources:]]
        
        # 3. MUSIC spectrum with finer resolution
        angles = np.linspace(-np.pi, np.pi, 720)  # 0.5° resolution
        spectrum = np.zeros(len(angles))
        
        for i, angle in enumerate(angles):
            # Steering vector for subarray dimensions
            phase = self.k * (pos_x_sub * np.cos(angle) + pos_y_sub * np.sin(angle))
            a = np.exp(1j * phase)
            denom = np.abs(a.conj() @ noise_subspace @ noise_subspace.conj().T @ a) + 1e-12
            spectrum[i] = 1.0 / denom
        
        # 4. Multi-peak detection for NLOS scenarios
        peaks, properties = find_peaks(spectrum, height=spectrum.max()*0.2, distance=20)
        
        if len(peaks) > 0:
            # If we have delay/amplitude info, prefer angle of first-arriving path
            if delays is not None and amplitudes is not None:
                powers = np.abs(amplitudes) ** 2
                power_threshold = powers.max() * 0.1
                valid_path_mask = powers > power_threshold
                
                if np.any(valid_path_mask):
                    # Get index of first significant path
                    valid_delays = np.where(valid_path_mask, delays, np.inf)
                    first_path_idx = np.argmin(valid_delays)
                    
                    # Weight peaks based on channel characteristics
                    peak_heights = spectrum[peaks]
                    # Select highest peak as primary estimate
                    best_peak = peaks[np.argmax(peak_heights)]
                else:
                    best_peak = peaks[np.argmax(spectrum[peaks])]
            else:
                best_peak = peaks[np.argmax(spectrum[peaks])]
            
            aoa_estimate = angles[best_peak]
        else:
            aoa_estimate = angles[np.argmax(spectrum)]
        
        # 5. Confidence metric (peak prominence)
        peak_prominence = spectrum.max() / (np.mean(spectrum) + 1e-12)
        
        if return_spectrum:
            return aoa_estimate, spectrum, peak_prominence
        else:
            return aoa_estimate, peak_prominence


class NLOSAwareBilaterationSolver:
    """Bilateration solver with NLOS-aware adaptive weighting"""
    @staticmethod
    def hybrid_localization_nlos_aware(anchor_positions, ranges, angles, 
                                        nlos_probs=None, distance_std=0.5, angle_std=0.1):
        """
        NLOS-aware hybrid localization with adaptive weighting.
        Reduces weight of measurements with high NLOS probability.
        """
        anchor_positions = np.asarray(anchor_positions)
        if anchor_positions.ndim == 1:
            anchor_positions = anchor_positions.reshape(1, -1)
        
        # Handle 3D anchor positions by flattening if needed
        if anchor_positions.ndim == 3:
            anchor_positions = anchor_positions.reshape(anchor_positions.shape[0], -1)
        
        ranges = np.asarray(ranges).flatten()
        angles = np.asarray(angles).flatten()
        
        if nlos_probs is None:
            nlos_probs = np.zeros(len(ranges))
        nlos_probs = np.asarray(nlos_probs).flatten()
        
        num_anchors = len(anchor_positions)
        
        def objective(pos):
            pos = np.asarray(pos).flatten()
            total_error = 0.0
            
            for i in range(num_anchors):
                # Adaptive weight based on NLOS probability (lower weight for NLOS)
                los_confidence = 1.0 - nlos_probs[i]
                
                # Distance error with NLOS-aware weighting
                distance_weight = los_confidence / (distance_std**2)
                anchor_pos = anchor_positions[i].flatten()[:len(pos)]
                dist = np.linalg.norm(pos - anchor_pos)
                total_error += distance_weight * (ranges[i] - dist)**2
                
                # Angle error with NLOS-aware weighting
                if not np.isnan(angles[i]):
                    angle_weight = los_confidence / (angle_std**2)
                    
                    pos_x = float(pos[0])
                    pos_y = float(pos[1])
                    anchor_x = float(anchor_pos[0])
                    anchor_y = float(anchor_pos[1])
                    
                    dir_x = pos_x - anchor_x
                    dir_y = pos_y - anchor_y
                    
                    estimated_angle = np.arctan2(dir_y, dir_x)
                    angle_diff = float(angles[i]) - estimated_angle
                    angle_diff = np.arctan2(np.sin(angle_diff), np.cos(angle_diff))
                    
                    total_error += angle_weight * (angle_diff**2)
            
            return float(total_error)
        
        x0 = np.mean(anchor_positions, axis=0).flatten()
        result = minimize(objective, x0, method='BFGS')
        return result.x


# --- IMPAIRMENT AND COMPENSATION FUNCTIONS ---

def add_sfo(csi_matrix, subcarrier_indices, sfo_ppm=5):
    """Adds Sampling Frequency Offset (SFO) to the CSI matrix."""
    clock_mismatch = sfo_ppm * 1e-6
    phase_slope = 2 * np.pi * clock_mismatch
    sfo_phase = phase_slope * subcarrier_indices
    sfo_matrix = np.tile(np.exp(1j * sfo_phase), (csi_matrix.shape[0], 1))
    return csi_matrix * sfo_matrix

def add_phase_noise(csi_matrix, pn_std=0.5):
    """Adds Phase Noise (PN) / Common Phase Error (CPE)."""
    pn_shift = np.random.normal(0, pn_std) 
    pn_factor = np.exp(1j * pn_shift)
    return csi_matrix * pn_factor

def compensate_phase_noise(csi_matrix):
    """Compensates for CPE by removing the mean phase rotation."""
    mean_complex = np.mean(csi_matrix)
    cpe = np.angle(mean_complex)
    return csi_matrix * np.exp(-1j * cpe)

def compensate_sfo(csi_matrix, subcarrier_indices):
    """Compensates for SFO by removing linear phase slope across subcarriers."""
    avg_csi = np.mean(csi_matrix, axis=0)
    avg_phase = np.unwrap(np.angle(avg_csi))
    coeffs = np.polyfit(subcarrier_indices, avg_phase, 1)
    sfo_compensation = np.exp(-1j * np.polyval(coeffs, subcarrier_indices))
    return csi_matrix * sfo_compensation

print("✓ All necessary classes and functions defined (including NLOS mitigation).")

✓ All necessary classes and functions defined (including NLOS mitigation).


In [3]:
# ============================================================================
# SCENE SETUP AND RAY TRACING
# ============================================================================

no_preview = False

mi.set_variant("llvm_ad_mono_polarized")

scene_path = "../scene/scene_02.xml"

print("Loading scene with Sionna...")
scene = load_scene(scene_path)
print("Sionna scene loaded.")

lambda_half = 0.0625  # half wavelength spacing at 2.4 GHz

# AP as RX (home router, omnidirectional)
scene.rx_array = PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=lambda_half,
    horizontal_spacing=lambda_half,
    pattern="dipole",
    polarization="V"
)

# STA as TX (mobile device)
scene.tx_array = PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=lambda_half,
    horizontal_spacing=lambda_half,
    pattern="dipole",
    polarization="cross"
)

# --- Transmitter & Receiver Positions ---
tx_positions = [
    np.array([0, 0.8, 0])
]
rx_positions = [
    np.array([1.5, 0.86, -1.15])
]

# --- Add TX/RX to Scene ---
tx_list, rx_list = [], []

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=f"tx_{i}", position=pos, display_radius=0.08)
    scene.add(tx)
    tx_list.append(tx)

for i, pos in enumerate(rx_positions):
    rx = Receiver(name=f"rx_{i}", position=pos, display_radius=0.08)
    scene.add(rx)
    rx_list.append(rx)

# Aim each transmitter toward each receiver
for tx in tx_list:
    for rx in rx_list:
        tx.look_at(rx)

print(f"Placed {len(tx_list)} transmitters and {len(rx_list)} receivers.\n")

camera_pos = [1.5, 1.0, 1.6]
camera_look = [1.5, 1.5, 1.3]

my_cam = Camera(
    position=camera_pos,
    look_at=camera_look,
)

solver_paths = PathSolver()
paths = solver_paths(scene, max_depth=5, los=True, specular_reflection=True, refraction=True)

num_paths = paths.tau.shape[-1]
print(f"Computed {num_paths} propagation paths per TX-RX link.\n")

Loading scene with Sionna...
Sionna scene loaded.
Placed 1 transmitters and 1 receivers.

Computed 90 propagation paths per TX-RX link.



In [5]:
# ============================================================================
# CSI CONVERSION, IMPAIRMENT FLOW, AND DERIVATION (WITH NLOS MITIGATION)
# ============================================================================

# --- Data Extraction from Sionna ---
a, tau = paths.cir(normalize_delays=False, out_type="numpy")
a_snap = a[..., 0] 
tau_snap = tau

# Dynamically determine dimensions from paths object
num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths = a_snap.shape

# Define Estimator/Solver instances (ORIGINAL - for comparison)
aoa_estimator = CSIAngleEstimator(num_rx_ant, WAVELENGTH)
range_estimator = FTMRangeEstimator()
solver = BilaterationSolver()

# Define ENHANCED Estimator/Solver instances (NLOS-MITIGATED)
enhanced_aoa_estimator = EnhancedAoAEstimator(num_rx_ant, WAVELENGTH)
enhanced_range_estimator = EnhancedFTMRangeEstimator()
nlos_solver = NLOSAwareBilaterationSolver()

# Choose which transmitter to localize (the mobile device)
target_tx_idx = 0  # Change this to localize a different TX

# Storage for ORIGINAL method results
all_estimated_ranges = []
all_estimated_angles = []
all_anchor_positions = []

# Storage for NLOS-MITIGATED method results
all_nlos_mitigated_ranges = []
all_nlos_mitigated_angles = []
all_nlos_probabilities = []
all_nlos_flags = []
all_nlos_metrics = []

true_target_position = tx_list[target_tx_idx].position.numpy().flatten()  # Flatten immediately

print("\n" + "=" * 70)
print(f"Localizing TX-{target_tx_idx} (Mobile Device) using RX measurements (APs)")
print(f"True TX-{target_tx_idx} Position: [{true_target_position[0]:.2f}, {true_target_position[1]:.2f}, {true_target_position[2]:.2f}]")
print("=" * 70)

# Process only links involving the target transmitter
for rx_idx in range(num_rx):
    rx_pos = rx_list[rx_idx].position.numpy()
    tx_pos = tx_list[target_tx_idx].position.numpy()
    
    # Calculate true AoA/Range (Ground Truth)
    # AoA is measured at RX (AP), pointing toward TX (mobile device)
    true_range = np.linalg.norm(tx_pos - rx_pos)
    direction = tx_pos[:2] - rx_pos[:2]  # Direction FROM RX TO TX
    true_angle = np.arctan2(direction[1], direction[0])
    
    # 1. GENERATE IDEAL CSI
    H_freq_base = np.zeros((NUM_SUBCARRIERS, num_rx_ant), dtype=np.complex128)
    
    for rx_ant in range(num_rx_ant):
        for tx_ant in range(num_tx_ant):
            # Fourier Conversion: CIR to CSI (preserve complex phase!)
            delays = tau_snap[rx_idx, target_tx_idx, :]
            complex_amplitudes = a_snap[rx_idx, rx_ant, target_tx_idx, tx_ant, :]  # Keep complex, not abs()
            for l in range(num_paths):
                H_freq_base[:, rx_ant] += complex_amplitudes[l] * np.exp(-1j * 2*np.pi * delays[l] * subcarriers)
    
    # Transpose CSI to [Rx Antennas, Subcarriers]
    csi_mimo = H_freq_base.T 

    # 2. ADD NON-IDEALITIES (IMPAIRMENT)
    csi_sfo = add_sfo(csi_mimo, subcarrier_indices, sfo_ppm=5)
    csi_pn = add_phase_noise(csi_sfo, pn_std=0.5)

    # Add AWGN (SNR = 20 dB)
    snr_db = 20
    signal_power = np.mean(np.abs(csi_pn)**2)
    noise_power = signal_power * 10**(-snr_db/10)
    noise = np.sqrt(noise_power/2) * (np.random.randn(*csi_pn.shape) + 1j * np.random.randn(*csi_pn.shape))
    final_csi = csi_pn + noise
    
    # 3. APPLY COMPENSATION (SFO + Phase Noise)
    csi_sfo_compensated = compensate_sfo(final_csi, subcarrier_indices)
    csi_compensated = compensate_phase_noise(csi_sfo_compensated)
    avg_csi_final = np.mean(csi_compensated, axis=0)
    
    # Get path delays and amplitudes for both methods
    delays = tau_snap[rx_idx, target_tx_idx, :]
    amplitudes = np.mean([a_snap[rx_idx, ant, target_tx_idx, 0, :] for ant in range(num_rx_ant)], axis=0)
    
    # =========================================================================
    # 4A. ORIGINAL METHOD (without NLOS mitigation)
    # =========================================================================
    estimated_angle, spectrum = aoa_estimator.estimate_aoa_music(csi_compensated, return_spectrum=True)
    peak_prominence = spectrum.max() / (np.mean(spectrum)+1e-12)
    if peak_prominence < 3:   # Lowered threshold to allow weaker but valid estimates
        estimated_angle = np.nan   # don't use a bad AoA
    
    estimated_range = range_estimator.estimate_range_from_cir(delays, amplitudes)

    # Store ORIGINAL results
    all_estimated_ranges.append(estimated_range)
    all_estimated_angles.append(float(estimated_angle))
    all_anchor_positions.append(rx_pos)  # RX (AP) is the anchor
    
    # =========================================================================
    # 4B. NLOS-MITIGATED METHOD
    # =========================================================================
    # Enhanced AoA estimation with spatial smoothing
    nlos_angle, nlos_spectrum, nlos_peak_prominence = enhanced_aoa_estimator.estimate_aoa_music_nlos_robust(
        csi_compensated, delays=delays, amplitudes=amplitudes, return_spectrum=True
    )
    
    # Use a lower threshold since spatial smoothing improves estimates
    if nlos_peak_prominence < 2:
        nlos_angle = np.nan
    
    # Enhanced range estimation with NLOS detection and mitigation
    nlos_range, is_nlos, nlos_prob = enhanced_range_estimator.estimate_range_nlos_mitigated(delays, amplitudes)
    
    # Get NLOS metrics for analysis
    _, _, nlos_metrics = enhanced_range_estimator.detect_nlos(delays, amplitudes)
    
    # Store NLOS-MITIGATED results
    all_nlos_mitigated_ranges.append(nlos_range)
    all_nlos_mitigated_angles.append(float(nlos_angle))
    all_nlos_probabilities.append(nlos_prob)
    all_nlos_flags.append(is_nlos)
    all_nlos_metrics.append(nlos_metrics)
    
    true_angle_scalar = float(true_angle)
    
    # Print comparison
    print(f"\nRX-{rx_idx} (AP) measuring TX-{target_tx_idx}:")
    print(f"  ORIGINAL:       Range={estimated_range:.2f}m (True={true_range:.2f}m), Angle={np.rad2deg(estimated_angle):.1f}°")
    print(f"  NLOS-MITIGATED: Range={nlos_range:.2f}m (True={true_range:.2f}m), Angle={np.rad2deg(nlos_angle):.1f}°")
    print(f"  NLOS Detection: is_NLOS={is_nlos}, probability={nlos_prob:.2f}")
    print(f"  NLOS Metrics: K-factor={nlos_metrics['k_factor']:.2f}, RMS delay spread={nlos_metrics['rms_delay_spread']*1e9:.2f}ns")
    print(f"  True Angle: {np.rad2deg(true_angle_scalar):.1f}°")


Localizing TX-0 (Mobile Device) using RX measurements (APs)
True TX-0 Position: [0.00, 0.80, 0.00]

RX-0 (AP) measuring TX-0:
  ORIGINAL:       Range=1.89m (True=1.89m), Angle=nan°
  NLOS-MITIGATED: Range=1.89m (True=1.89m), Angle=nan°
  NLOS Detection: is_NLOS=False, probability=0.30
  NLOS Metrics: K-factor=10.36, RMS delay spread=2.41ns
  True Angle: -177.7°


/var/folders/89/plsvkd5n25dg_s6jx37w0sn00000gn/T/ipykernel_49022/2501785554.py:130: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  true_angle_scalar = float(true_angle)


In [6]:
# ============================================================================
# FINAL LOCALIZATION DERIVATION AND RESULTS (ORIGINAL vs NLOS-MITIGATED)
# ============================================================================

print("\n" + "=" * 70)
print(f"Performing Hybrid Bilateration to Localize TX-{target_tx_idx}")
print("=" * 70)

# Convert results to NumPy arrays
anchor_pos_array_final = np.array(all_anchor_positions)
ranges_final = np.array(all_estimated_ranges)
angles_final = np.array(all_estimated_angles)

# NLOS-mitigated results
nlos_ranges_final = np.array(all_nlos_mitigated_ranges)
nlos_angles_final = np.array(all_nlos_mitigated_angles)
nlos_probs_final = np.array(all_nlos_probabilities)

# Debug information
print(f"\nInput Data Summary:")
print(f"  Number of anchors (RX/APs): {len(anchor_pos_array_final)}")
print(f"  Anchor positions shape: {anchor_pos_array_final.shape}")
print(f"  Original - Ranges: {ranges_final}, Angles: {np.rad2deg(angles_final)}°")
print(f"  NLOS-Mit - Ranges: {nlos_ranges_final}, Angles: {np.rad2deg(nlos_angles_final)}°")
print(f"  NLOS Probabilities: {nlos_probs_final}")

# Flatten target position
true_target_position_flat = np.asarray(true_target_position).flatten()

# =========================================================================
# ORIGINAL HYBRID LOCALIZATION
# =========================================================================
try:
    estimated_position = solver.hybrid_localization(
        anchor_pos_array_final,
        ranges_final,
        angles_final,
        distance_std=0.5, 
        angle_std=np.deg2rad(5)
    )
    
    estimated_position = np.asarray(estimated_position).flatten()
    localization_error = np.linalg.norm(estimated_position - true_target_position_flat)
    error_xyz = estimated_position - true_target_position_flat
    
    print(f"\n{'='*60}")
    print(f"ORIGINAL HYBRID LOCALIZATION RESULTS FOR TX-{target_tx_idx}")
    print(f"{'='*60}")
    print(f"True TX Position:      [{true_target_position_flat[0]:.2f}, {true_target_position_flat[1]:.2f}, {true_target_position_flat[2]:.2f}]")
    print(f"Estimated TX Position: [{estimated_position[0]:.2f}, {estimated_position[1]:.2f}, {estimated_position[2]:.2f}]")
    print(f"Localization Error: {localization_error:.3f} meters")
    
    print(f"\nPer-Dimension Errors:")
    print(f"  X-axis error: {error_xyz[0]:+.3f} m")
    print(f"  Y-axis error: {error_xyz[1]:+.3f} m")
    print(f"  Z-axis error: {error_xyz[2]:+.3f} m")
    
except Exception as e:
    print(f"\n❌ Original localization failed: {type(e).__name__}: {str(e)}")
    estimated_position = None
    localization_error = np.inf

# =========================================================================
# NLOS-MITIGATED HYBRID LOCALIZATION
# =========================================================================
try:
    estimated_position_nlos = nlos_solver.hybrid_localization_nlos_aware(
        anchor_pos_array_final,
        nlos_ranges_final,
        nlos_angles_final,
        nlos_probs=nlos_probs_final,
        distance_std=0.5, 
        angle_std=np.deg2rad(5)
    )
    
    estimated_position_nlos = np.asarray(estimated_position_nlos).flatten()
    localization_error_nlos = np.linalg.norm(estimated_position_nlos - true_target_position_flat)
    error_xyz_nlos = estimated_position_nlos - true_target_position_flat
    
    print(f"\n{'='*60}")
    print(f"NLOS-MITIGATED HYBRID LOCALIZATION RESULTS FOR TX-{target_tx_idx}")
    print(f"{'='*60}")
    print(f"True TX Position:      [{true_target_position_flat[0]:.2f}, {true_target_position_flat[1]:.2f}, {true_target_position_flat[2]:.2f}]")
    print(f"Estimated TX Position: [{estimated_position_nlos[0]:.2f}, {estimated_position_nlos[1]:.2f}, {estimated_position_nlos[2]:.2f}]")
    print(f"Localization Error: {localization_error_nlos:.3f} meters")
    
    print(f"\nPer-Dimension Errors:")
    print(f"  X-axis error: {error_xyz_nlos[0]:+.3f} m")
    print(f"  Y-axis error: {error_xyz_nlos[1]:+.3f} m")
    print(f"  Z-axis error: {error_xyz_nlos[2]:+.3f} m")
    
except Exception as e:
    print(f"\n❌ NLOS-mitigated localization failed: {type(e).__name__}: {str(e)}")
    estimated_position_nlos = None
    localization_error_nlos = np.inf

# =========================================================================
# COMPARISON: ORIGINAL vs NLOS-MITIGATED
# =========================================================================
print(f"\n{'='*60}")
print(f"COMPARISON: ORIGINAL vs NLOS-MITIGATED HYBRID")
print(f"{'='*60}")

if localization_error != np.inf and localization_error_nlos != np.inf:
    improvement = localization_error - localization_error_nlos
    improvement_pct = (improvement / localization_error) * 100 if localization_error > 0 else 0
    
    print(f"\n  {'Method':<25} {'Error (m)':<12} {'X-Err (m)':<12} {'Y-Err (m)':<12} {'Z-Err (m)':<12}")
    print(f"  {'-'*73}")
    print(f"  {'Original Hybrid':<25} {localization_error:<12.3f} {error_xyz[0]:<+12.3f} {error_xyz[1]:<+12.3f} {error_xyz[2]:<+12.3f}")
    print(f"  {'NLOS-Mitigated Hybrid':<25} {localization_error_nlos:<12.3f} {error_xyz_nlos[0]:<+12.3f} {error_xyz_nlos[1]:<+12.3f} {error_xyz_nlos[2]:<+12.3f}")
    
    print(f"\n  Improvement from NLOS Mitigation:")
    print(f"    Error Reduction: {improvement:+.3f} meters")
    print(f"    Improvement:     {improvement_pct:+.1f}%")
    
    if localization_error_nlos < localization_error:
        print(f"    ✓ NLOS mitigation IMPROVED accuracy!")
    elif localization_error_nlos > localization_error:
        print(f"    ⚠ NLOS mitigation did not improve accuracy in this case")
    else:
        print(f"    = Both methods have equal performance")

# Individual link statistics
print(f"\n  Individual Link Performance (Range Errors):")
for i in range(len(anchor_pos_array_final)):
    anchor_pos = anchor_pos_array_final[i].flatten()
    true_range = np.linalg.norm(true_target_position_flat - anchor_pos[:len(true_target_position_flat)])
    
    orig_range_error = abs(float(ranges_final[i]) - float(true_range))
    nlos_range_error = abs(float(nlos_ranges_final[i]) - float(true_range))
    
    print(f"    RX-{i}: Original={orig_range_error:.3f}m, NLOS-Mit={nlos_range_error:.3f}m, NLOS_prob={nlos_probs_final[i]:.2f}")

print(f"{'='*60}")


Performing Hybrid Bilateration to Localize TX-0

Input Data Summary:
  Number of anchors (RX/APs): 1
  Anchor positions shape: (1, 3, 1)
  Original - Ranges: [1.89236693], Angles: [nan]°
  NLOS-Mit - Ranges: [1.89236693], Angles: [nan]°
  NLOS Probabilities: [0.3]

ORIGINAL HYBRID LOCALIZATION RESULTS FOR TX-0
True TX Position:      [0.00, 0.80, 0.00]
Estimated TX Position: [0.48, 0.47, 0.15]
Localization Error: 0.604 meters

Per-Dimension Errors:
  X-axis error: +0.482 m
  Y-axis error: -0.332 m
  Z-axis error: +0.149 m

NLOS-MITIGATED HYBRID LOCALIZATION RESULTS FOR TX-0
True TX Position:      [0.00, 0.80, 0.00]
Estimated TX Position: [2.59, 1.95, -2.24]
Localization Error: 3.616 meters

Per-Dimension Errors:
  X-axis error: +2.593 m
  Y-axis error: +1.153 m
  Z-axis error: -2.241 m

COMPARISON: ORIGINAL vs NLOS-MITIGATED HYBRID

  Method                    Error (m)    X-Err (m)    Y-Err (m)    Z-Err (m)   
  -------------------------------------------------------------------------

In [7]:
# ============================================================================
# RANGE-ONLY LOCALIZATION (RTT/FTM WITHOUT AoA)
# ============================================================================

print("\n" + "=" * 70)
print(f"Performing Range-Only Localization (RTT/FTM only, no AoA)")
print("=" * 70)

# Simulate RTT-only measurements with timing impairments
# RTT is affected by: clock drift, processing delays, multipath (less than CSI)
rtt_ranges = []

print("\nSimulating RTT-only measurements:")
for i, (rx_pos, true_range) in enumerate(zip(anchor_pos_array_final, ranges_final)):
    # Calculate true range
    true_rtt_range = np.linalg.norm(true_target_position - rx_pos)
    
    # Add RTT-specific impairments
    # 1. Clock drift/offset (systematic error) - typically 10-50 ns
    clock_offset_ns = np.random.normal(0, 20)  # 20 ns std dev
    clock_error_m = (clock_offset_ns * 1e-9) * 3e8 / 2  # Convert to meters (two-way)
    
    # 2. Processing delay variation (random per measurement)
    processing_delay_ns = np.random.uniform(0, 10)  # 0-10 ns
    processing_error_m = (processing_delay_ns * 1e-9) * 3e8 / 2
    
    # 3. Multipath adds positive bias (signal takes longer path)
    multipath_bias_m = np.random.exponential(0.3)  # Exponential distribution, mean 0.3m
    
    # 4. Measurement noise (AWGN on time-of-arrival)
    noise_std_ns = 5  # 5 ns noise standard deviation
    noise_m = np.random.normal(0, noise_std_ns * 1e-9 * 3e8 / 2)
    
    # Total RTT range estimate with impairments
    rtt_range = true_rtt_range + clock_error_m + processing_error_m + multipath_bias_m + noise_m
    rtt_ranges.append(rtt_range)
    
    print(f"  RX-{i}: True={true_rtt_range:.3f}m, RTT Est={rtt_range:.3f}m, Error={rtt_range-true_rtt_range:+.3f}m")

rtt_ranges = np.array(rtt_ranges)

# Use NaN for all angles (no angle information in RTT-only)
angles_nan = np.full_like(angles_final, np.nan)

try:
    estimated_position_rtt_only = solver.hybrid_localization(
        anchor_pos_array_final,
        rtt_ranges,  # Use RTT-specific range estimates
        angles_nan,  # No angle information
        distance_std=0.5, 
        angle_std=np.deg2rad(5)
    )
    
    # Flatten arrays
    estimated_position_rtt_only = np.asarray(estimated_position_rtt_only).flatten()
    
    # Calculate error
    localization_error_rtt_only = np.linalg.norm(estimated_position_rtt_only - true_target_position)
    
    print(f"\n{'='*60}")
    print(f"RTT-ONLY LOCALIZATION RESULTS FOR TX-{target_tx_idx}")
    print(f"{'='*60}")
    print(f"True TX Position:           [{true_target_position[0]:.2f}, {true_target_position[1]:.2f}, {true_target_position[2]:.2f}]")
    print(f"Estimated TX Position:      [{estimated_position_rtt_only[0]:.2f}, {estimated_position_rtt_only[1]:.2f}, {estimated_position_rtt_only[2]:.2f}]")
    print(f"Localization Error:         {localization_error_rtt_only:.3f} meters")
    
    # Per-dimension errors
    error_xyz_rtt = estimated_position_rtt_only - true_target_position
    print(f"\nPer-Dimension Errors:")
    print(f"  X-axis error: {error_xyz_rtt[0]:+.3f} m")
    print(f"  Y-axis error: {error_xyz_rtt[1]:+.3f} m")
    print(f"  Z-axis error: {error_xyz_rtt[2]:+.3f} m")
    print(f"{'='*60}")
    
except Exception as e:
    print(f"\n❌ RTT-only localization failed with error:")
    print(f"   {type(e).__name__}: {str(e)}")
    raise


Performing Range-Only Localization (RTT/FTM only, no AoA)

Simulating RTT-only measurements:
  RX-0: True=3.594m, RTT Est=8.012m, Error=+4.417m

RTT-ONLY LOCALIZATION RESULTS FOR TX-0
True TX Position:           [0.00, 0.80, 0.00]
Estimated TX Position:      [2.72, 1.39, -2.95]
Localization Error:         4.054 meters

Per-Dimension Errors:
  X-axis error: +2.721 m
  Y-axis error: +0.593 m
  Z-axis error: -2.946 m


In [8]:
# ============================================================================
# COMPREHENSIVE COMPARISON: ALL METHODS
# ============================================================================

print("\n" + "=" * 70)
print("COMPREHENSIVE COMPARISON: ALL LOCALIZATION METHODS")
print("=" * 70)

# Calculate true ranges for error computation
true_ranges_list = []
for i in range(len(anchor_pos_array_final)):
    anchor_pos = anchor_pos_array_final[i].flatten()
    tr = np.linalg.norm(true_target_position - anchor_pos[:len(true_target_position)])
    true_ranges_list.append(tr)

# Range errors for each method
orig_range_errors = [abs(float(ranges_final[i]) - true_ranges_list[i]) for i in range(len(ranges_final))]
nlos_range_errors = [abs(float(nlos_ranges_final[i]) - true_ranges_list[i]) for i in range(len(nlos_ranges_final))]
rtt_range_errors = [abs(float(rtt_ranges[i]) - true_ranges_list[i]) for i in range(len(rtt_ranges))]

print(f"\n{'='*80}")
print(f"LOCALIZATION ACCURACY COMPARISON")
print(f"{'='*80}")
print(f"\n  {'Method':<30} {'Error (m)':<12} {'X-Err (m)':<12} {'Y-Err (m)':<12} {'Z-Err (m)':<12}")
print(f"  {'-'*78}")
print(f"  {'Original Hybrid':<30} {localization_error:<12.3f} {error_xyz[0]:<+12.3f} {error_xyz[1]:<+12.3f} {error_xyz[2]:<+12.3f}")
print(f"  {'NLOS-Mitigated Hybrid':<30} {localization_error_nlos:<12.3f} {error_xyz_nlos[0]:<+12.3f} {error_xyz_nlos[1]:<+12.3f} {error_xyz_nlos[2]:<+12.3f}")
print(f"  {'RTT-Only (Time-of-Flight)':<30} {localization_error_rtt_only:<12.3f} {error_xyz_rtt[0]:<+12.3f} {error_xyz_rtt[1]:<+12.3f} {error_xyz_rtt[2]:<+12.3f}")

# Find best method
errors = {
    'Original Hybrid': localization_error,
    'NLOS-Mitigated Hybrid': localization_error_nlos,
    'RTT-Only': localization_error_rtt_only
}
best_method = min(errors, key=errors.get)
print(f"\n  ★ Best Method: {best_method} (Error: {errors[best_method]:.3f}m)")

# NLOS mitigation improvement analysis
print(f"\n{'='*80}")
print(f"NLOS MITIGATION IMPROVEMENT ANALYSIS")
print(f"{'='*80}")

nlos_improvement = localization_error - localization_error_nlos
nlos_improvement_pct = (nlos_improvement / localization_error) * 100 if localization_error > 0 else 0

print(f"\n  NLOS Mitigation Effect on Hybrid Method:")
print(f"    Original Error:        {localization_error:.3f} m")
print(f"    NLOS-Mitigated Error:  {localization_error_nlos:.3f} m")
print(f"    Error Reduction:       {nlos_improvement:+.3f} m")
print(f"    Improvement:           {nlos_improvement_pct:+.1f}%")

if nlos_improvement > 0:
    print(f"    ✓ NLOS mitigation IMPROVED localization accuracy!")
elif nlos_improvement < 0:
    print(f"    ⚠ NLOS mitigation did not help in this scenario")
else:
    print(f"    = No change from NLOS mitigation")

# Range estimation comparison
print(f"\n{'='*80}")
print(f"RANGE ESTIMATION ACCURACY")
print(f"{'='*80}")
print(f"\n  {'Method':<30} {'Avg Range Error (m)':<20} {'Max Range Error (m)':<20}")
print(f"  {'-'*70}")
print(f"  {'Original CSI-based':<30} {np.mean(orig_range_errors):<20.3f} {np.max(orig_range_errors):<20.3f}")
print(f"  {'NLOS-Mitigated CSI':<30} {np.mean(nlos_range_errors):<20.3f} {np.max(nlos_range_errors):<20.3f}")
print(f"  {'RTT-based':<30} {np.mean(rtt_range_errors):<20.3f} {np.max(rtt_range_errors):<20.3f}")

# NLOS detection summary
print(f"\n{'='*80}")
print(f"NLOS DETECTION SUMMARY")
print(f"{'='*80}")
print(f"\n  Per-Link NLOS Analysis:")
for i in range(len(all_nlos_flags)):
    metrics = all_nlos_metrics[i]
    print(f"    RX-{i}: NLOS={all_nlos_flags[i]}, Prob={all_nlos_probabilities[i]:.2f}")
    print(f"           K-factor={metrics['k_factor']:.2f}, Delay Spread={metrics['rms_delay_spread']*1e9:.2f}ns")
    print(f"           First-Strongest Delay Diff={metrics['delay_diff']*1e9:.2f}ns")

# Final analysis notes
print(f"\n{'='*80}")
print("ANALYSIS NOTES:")
print("="*80)
print("""
METHODS COMPARED:
  1. Original Hybrid: CSI-based ranging + MUSIC AoA (baseline)
  2. NLOS-Mitigated Hybrid: Enhanced with:
     - NLOS detection (Kurtosis, K-factor, delay spread)
     - First-arrival ranging with stricter thresholds
     - NLOS bias correction
     - Spatial smoothing for AoA (coherent multipath handling)
     - NLOS-aware weighting in optimization
  3. RTT-Only: Time-of-flight ranging (no AoA)

KEY INSIGHTS:
  - NLOS mitigation helps when multipath/reflections dominate
  - Spatial smoothing improves AoA estimation in correlated multipath
  - NLOS-aware weighting reduces influence of biased measurements
  - In strong LOS conditions, NLOS mitigation may show minimal improvement
  - More anchors would improve all methods significantly
""")
print(f"{'='*80}")


COMPREHENSIVE COMPARISON: ALL LOCALIZATION METHODS

LOCALIZATION ACCURACY COMPARISON

  Method                         Error (m)    X-Err (m)    Y-Err (m)    Z-Err (m)   
  ------------------------------------------------------------------------------
  Original Hybrid                0.604        +0.482       -0.332       +0.149      
  NLOS-Mitigated Hybrid          3.616        +2.593       +1.153       -2.241      
  RTT-Only (Time-of-Flight)      4.054        +2.721       +0.593       -2.946      

  ★ Best Method: Original Hybrid (Error: 0.604m)

NLOS MITIGATION IMPROVEMENT ANALYSIS

  NLOS Mitigation Effect on Hybrid Method:
    Original Error:        0.604 m
    NLOS-Mitigated Error:  3.616 m
    Error Reduction:       -3.013 m
    Improvement:           -498.8%
    ⚠ NLOS mitigation did not help in this scenario

RANGE ESTIMATION ACCURACY

  Method                         Avg Range Error (m)  Max Range Error (m) 
  --------------------------------------------------------------